### Links and Resources
| Concept | Reference |
|---|---|
| [Python UDFs in UC](https://docs.databricks.com/aws/en/udf/unity-catalog-python-udfs) | DDL & runtime model |
| [`CREATE FUNCTION` Python form](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-python-udf) | Syntax |
| [Allowed packages](https://docs.databricks.com/aws/en/udf/unity-catalog-python-udfs#dependencies) | What's importable |
| [Sandboxing](https://docs.databricks.com/aws/en/udf/unity-catalog-python-udfs#security) | Network/file restrictions |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Python UDFs in Unity Catalog

A UC Python UDF is a Python function whose body lives **inside the catalog**. Unlike a notebook-scoped UDF (notebook 11), it:

- Is callable from **any** session — Python, SQL warehouse, BI tool — that has `EXECUTE`.
- Survives notebook restarts; the source is stored in UC.
- Runs in a **sandboxed environment** with no network and a curated set of standard-library + data-science packages.
- Is governed by `GRANT` / `REVOKE` like a table or SQL UDF.

The trade-off: tighter sandbox, fewer packages, and the same per-row Python overhead as a notebook UDF — so prefer SQL UDFs (notebook 13) when the logic fits.

## Why this matters

Most projects want one of:

1. **Reusable logic across many notebooks/dashboards** without copy-paste.
2. **A single auditable definition** of a business rule.
3. **A safe way to share Python helpers** with SQL-only users.

A UC Python UDF gives you all three. It's the production-friendly cousin of the in-notebook UDF.

In [ ]:
CATALOG = "main"
SCHEMA  = "default"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA  {SCHEMA}")
print("using:", spark.sql("SELECT current_catalog(), current_schema()").collect()[0])

## Create a Python UDF in UC

The body is **Python code embedded in the SQL `CREATE FUNCTION` statement**. The function name in the `$$ ... $$` block must match the SQL function name.

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION normalize_phone(raw STRING)
    RETURNS STRING
    LANGUAGE PYTHON
    COMMENT 'Strip non-digits, keep last 10, format as (XXX) XXX-XXXX. NULL on bad input.'
    AS $$
        if raw is None:
            return None
        digits = ''.join(c for c in raw if c.isdigit())
        if len(digits) < 10:
            return None
        d = digits[-10:]
        return f"({d[0:3]}) {d[3:6]}-{d[6:10]}"
    $$
""")
print("created normalize_phone")

In [ ]:
spark.sql("""
SELECT raw, normalize_phone(raw) AS clean FROM (VALUES
    ('+1 (415) 555-0123'),
    ('415.555.0123'),
    ('555-0123'),                  -- too short
    (NULL),
    ('abc')                        -- no digits
) AS t(raw)
""").show(truncate=False)

## Multi-argument UDF — currency conversion stub

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION convert_amount(amount DOUBLE, from_ccy STRING, to_ccy STRING)
    RETURNS DOUBLE
    LANGUAGE PYTHON
    AS $$
        # Hard-coded rate table for the demo. In production, pull from a UC table.
        rates = {
            ('USD', 'EUR'): 0.92,
            ('EUR', 'USD'): 1.09,
            ('USD', 'GBP'): 0.79,
            ('GBP', 'USD'): 1.27,
        }
        if amount is None or from_ccy is None or to_ccy is None:
            return None
        if from_ccy == to_ccy:
            return amount
        rate = rates.get((from_ccy, to_ccy))
        return None if rate is None else round(amount * rate, 2)
    $$
""")

spark.sql("""
SELECT * FROM (VALUES
    (100.0, 'USD', 'EUR'),
    (50.0,  'EUR', 'USD'),
    (10.0,  'USD', 'JPY')          -- unknown rate
) AS t(amount, from_ccy, to_ccy)
""").withColumn("converted", F.expr("convert_amount(amount, from_ccy, to_ccy)")).show()

## Returning structured types

Like SQL UDFs, the return type can be `ARRAY<...>`, `MAP<...>`, or `STRUCT<...>`. Inside Python, return a list / dict / tuple.

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION parse_name(full STRING)
    RETURNS STRUCT<first STRING, last STRING, initials STRING>
    LANGUAGE PYTHON
    AS $$
        if not full:
            return None
        parts = full.strip().split()
        if not parts:
            return None
        first = parts[0]
        last  = parts[-1] if len(parts) > 1 else None
        initials = ''.join(p[0].upper() for p in parts if p)
        return (first, last, initials)
    $$
""")

spark.sql("""
SELECT parse_name(n).* FROM (VALUES ('Alice Cooper'), ('Charlie'), ('Dana O''Neill'), (NULL)) AS t(n)
""").show()

## Allowed dependencies

The sandbox ships with the Python standard library plus a curated set of data-science packages: `numpy`, `pandas`, `scipy`, `scikit-learn`, `cryptography`, etc. The exact list depends on your DBR — check the docs link at the top.

**No network**, **no arbitrary file system**, **no `pip install`**. If you need a package not on the list, use a notebook-scoped UDF (notebook 11) or a Pandas UDF on a regular cluster (notebook 12).

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION standardize(value DOUBLE)
    RETURNS DOUBLE
    LANGUAGE PYTHON
    AS $$
        import math
        # Hard-coded population stats for the demo
        mu, sigma = 50.0, 10.0
        if value is None or sigma == 0:
            return None
        return (value - mu) / sigma
    $$
""")

spark.sql("SELECT standardize(v) FROM (VALUES (60.0), (40.0), (70.0)) t(v)").show()

## Permissions — same model as SQL UDFs

In [ ]:
# spark.sql("GRANT EXECUTE ON FUNCTION normalize_phone TO `analysts@example.com`")
# spark.sql("GRANT USAGE   ON SCHEMA   default        TO `analysts@example.com`")
# spark.sql("GRANT USAGE   ON CATALOG  main           TO `analysts@example.com`")
spark.sql("SHOW USER FUNCTIONS LIKE 'normalize_phone'").show(truncate=False)
spark.sql("DESCRIBE FUNCTION EXTENDED normalize_phone").show(20, truncate=False)

## Performance

A UC Python UDF runs row-by-row through the Python sandbox — same overhead profile as a classic Python UDF. **Don't** use it as a hot-path transform on a billion rows.

It's the right choice when:

- The logic must be **shared across many queries / users / tools**.
- A SQL UDF (notebook 13) can't express it.
- The data volume is moderate, or the UDF is a small part of a heavier query.

For high-throughput vectorized work, stay on a regular cluster and use Pandas UDFs (notebook 12).

## Cleanup

In [ ]:
spark.sql("DROP FUNCTION IF EXISTS normalize_phone")
spark.sql("DROP FUNCTION IF EXISTS convert_amount")
spark.sql("DROP FUNCTION IF EXISTS parse_name")
spark.sql("DROP FUNCTION IF EXISTS standardize")
print("dropped")

## Takeaways

- UC Python UDFs are governed, sharable Python functions stored in Unity Catalog.
- Body is embedded Python (`$$ ... $$`); name in the body must match the SQL name.
- Sandbox: standard library + curated data-science stack, **no network**, **no pip**.
- Same row-by-row overhead as a notebook Python UDF — pick **SQL UDF first**, then UC Python UDF if SQL can't do it.
- Permissions: `GRANT EXECUTE` on the function plus `USAGE` on the parent schema/catalog.